# Week 5 Exercise — RAG for West Africa (Côte d'Ivoire focus)

An **Expert Knowledge Worker** built with the full Week 5 curriculum: **RAG (Retrieval Augmented Generation)** tailored to **Africa, especially Ivory Coast (Côte d'Ivoire) and West Africa**. The assistant answers questions on **agriculture** (cocoa, rice, food security), **health** (malaria, maternal and child health), and **finance** (mobile money, microfinance) using a curated knowledge base.

**Week 5 techniques used:**  
- **Day 2:** Chunking (RecursiveCharacterTextSplitter), metadata by category (doc_type), embeddings (OpenAI or **HuggingFace** for low-cost), ChromaDB.  
- **Day 3:** Retriever + LLM, system prompt with context, Gradio ChatInterface.  
- **Day 4:** Simple evaluation idea (test question).  
- **Day 5:** Structured knowledge base by domain (agriculture / health / finance).

**Bilingual:** Answer in the user's language (French or English).  

Run from **repo root** or **community-contributions/asket/week5/** — the notebook uses the local `knowledge-base-africa/` folder.

In [1]:
# Imports (LangChain 1.0, aligned with Week 5)

import os
import glob
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

load_dotenv(override=True)

# Optional: HuggingFace embeddings (low-cost, useful when API budget is limited)
try:
    from langchain_huggingface import HuggingFaceEmbeddings
    HAS_HF = True
except ImportError:
    HAS_HF = False

## Paths and API config

Resolve **knowledge-base-africa** (West Africa / Côte d'Ivoire docs). Choose **OpenAI** or **HuggingFace** embeddings (HuggingFace = low-cost, no API key for embeddings). Chat: **OpenRouter** or **OpenAI**.

In [2]:
# 1) Knowledge base: prefer knowledge-base-africa (West Africa); fallback to course week5/knowledge-base
def find_kb():
    cwd = os.getcwd()
    # Next to this notebook (community-contributions/asket/week5/)
    local_africa = os.path.join(cwd, "knowledge-base-africa")
    if os.path.isdir(local_africa):
        return local_africa
    # Walk up to find Africa KB (e.g. if run from repo root)
    d = cwd
    for _ in range(10):
        kb = os.path.join(d, "community-contributions", "asket", "week5", "knowledge-base-africa")
        if os.path.isdir(kb):
            return kb
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    # Fallback: course knowledge-base (when notebook is in week5/community-contributions/asket/)
    course_kb = os.path.join(cwd, "..", "knowledge-base")
    if os.path.isdir(os.path.normpath(course_kb)):
        return os.path.normpath(course_kb)
    d = cwd
    for _ in range(10):
        course_kb = os.path.join(d, "week5", "knowledge-base")
        if os.path.isdir(course_kb):
            return course_kb
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return local_africa  # for error message

KB_PATH = find_kb()
if not os.path.isdir(KB_PATH):
    raise FileNotFoundError(f"Knowledge base not found: {KB_PATH}. Add knowledge-base-africa/ or run from week5/community-contributions/asket/ (course KB).")
print(f"Using knowledge base: {KB_PATH}")

DB_NAME = "vector_db_asket_africa"

# 2) Embeddings: HuggingFace (low-cost, no key) or OpenAI (set OPENAI_API_KEY)
openai_key = os.getenv("OPENAI_API_KEY")
USE_HF_EMBEDDINGS = not openai_key or os.getenv("USE_HF_EMBEDDINGS", "").lower() in ("1", "true", "yes")
if USE_HF_EMBEDDINGS and HAS_HF:
    EMBEDDINGS = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    print("Embeddings: HuggingFace (all-MiniLM-L6-v2) — low-cost.")
else:
    if not openai_key:
        print("Warning: OPENAI_API_KEY not set; falling back to HuggingFace if available.")
        if HAS_HF:
            EMBEDDINGS = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        else:
            raise RuntimeError("Set OPENAI_API_KEY or install langchain-huggingface.")
    else:
        EMBEDDINGS = OpenAIEmbeddings()
        print("Embeddings: OpenAI.")

# 3) Chat: OpenRouter preferred, fallback OpenAI
openrouter_key = os.getenv("OPENROUTER_API_KEY")
if openrouter_key and (openrouter_key.startswith("sk-or-") or openrouter_key.startswith("sk-proj-")):
    CHAT_MODEL = "openai/gpt-4o-mini"
    CHAT_BASE_URL = "https://openrouter.ai/api/v1"
    CHAT_API_KEY = openrouter_key
    print("Chat: OpenRouter.")
elif openai_key:
    CHAT_MODEL = "gpt-4o-mini"
    CHAT_BASE_URL = None
    CHAT_API_KEY = openai_key
    print("Chat: OpenAI.")
else:
    CHAT_MODEL = "gpt-4o-mini"
    CHAT_BASE_URL = None
    CHAT_API_KEY = None
    print("Chat: set OPENROUTER_API_KEY or OPENAI_API_KEY for chat.")

Using knowledge base: /Users/franckasket/Documents/GitHub/llm_engineering/community-contributions/asket/week5/knowledge-base-africa
Embeddings: OpenAI.
Chat: OpenRouter.


## Part A — Chunking (Day 2 style)

Load markdown docs from **knowledge-base-africa** by category (agriculture, health, finance). Add **doc_type** metadata for each document, then split with `RecursiveCharacterTextSplitter` and store in Chroma. Re-run to rebuild the index after adding or editing docs.

In [3]:
# Load by subfolder and tag each doc with doc_type (Day 2 pattern)
folders = [f for f in glob.glob(os.path.join(KB_PATH, "*")) if os.path.isdir(f)]
# Also include .md files at KB root (e.g. context-cote-d-ivoire.md)
documents = []
for folder in sorted(folders):
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
    )
    for doc in loader.load():
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)
# Root-level .md
root_loader = DirectoryLoader(
    KB_PATH, glob="*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
)
for doc in root_loader.load():
    doc.metadata["doc_type"] = "context"
    documents.append(doc)

print(f"Loaded {len(documents)} documents (categories: {set(d.metadata.get('doc_type') for d in documents)}).")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")

vectorstore = Chroma.from_documents(chunks, EMBEDDINGS, persist_directory=DB_NAME)
print(f"Vector store saved to {DB_NAME}.")

Loaded 7 documents (categories: {'health', 'finance', 'agriculture', 'context'}).
Split into 30 chunks.
Vector store saved to vector_db_asket_africa.


## Part B — Retriever and LLM (Day 3)

Connect to the persisted Chroma store, create a retriever (top-k), and set up the chat model for generation.

In [4]:
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=EMBEDDINGS)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

kwargs = {"model": CHAT_MODEL, "temperature": 0}
if CHAT_BASE_URL:
    kwargs["base_url"] = CHAT_BASE_URL
if CHAT_API_KEY:
    kwargs["api_key"] = CHAT_API_KEY
llm = ChatOpenAI(**kwargs)

## RAG answer function (Day 3)

Retrieve top-k chunks, format as context, and call the LLM. **Bilingual:** answer in the same language as the user (French or English).

In [5]:
SYSTEM_PROMPT_TEMPLATE = """
You are a helpful Expert Knowledge Worker for West Africa, with a focus on Côte d'Ivoire (Ivory Coast).
You answer questions using ONLY the following context from a curated knowledge base on agriculture, health, and finance in the region.
If the context does not contain relevant information, say so clearly. Be concise and accurate.
IMPORTANT: Answer in the SAME LANGUAGE as the user's question (French or English). If the user writes in French, answer in French; if in English, answer in English.

Context:
{context}
"""

def answer_question(question: str) -> str:
    if not question or not question.strip():
        return "Please ask a question about agriculture, health, or finance in Côte d'Ivoire / West Africa. / Posez une question sur l'agriculture, la santé ou la finance en Côte d'Ivoire / Afrique de l'Ouest."
    docs = retriever.invoke(question.strip())
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question.strip()),
    ])
    return response.content

In [6]:
# Quick test (English and French)
print(answer_question("What are the main mobile money providers in Côte d'Ivoire?"))
print("---")
print(answer_question("Quelles sont les principales mesures de prévention du paludisme en Côte d'Ivoire ?"))

The main mobile money providers in Côte d'Ivoire are:

- **Orange Money:** Offered by Orange, used for person-to-person transfers, bill payments, merchant payments, and airtime.
- **MTN MoMo (MTN Mobile Money):** MTN's mobile money service, similar in uses to Orange Money.
- **Wave:** A mobile money and payment app that has expanded in Côte d'Ivoire, often with low or no fees for basic transfers.
- **Moov Money:** Another operator offering mobile money services in the region.

Other operators also provide mobile money services in Côte d'Ivoire.
---
Les principales mesures de prévention du paludisme en Côte d'Ivoire sont :

- **Moustiquaires imprégnées d'insecticide à longue durée d'action (MILDA) :** Dormir sous une moustiquaire traitée avec insecticide chaque nuit est l'une des mesures les plus efficaces. Les moustiquaires sont souvent distribuées lors de campagnes nationales, dans les établissements de santé et les écoles.

- **Pulvérisation intradomiciliaire (PDI) :** Dans certaines

## Simple check (Day 4 — evaluation idea)

Run one question and optionally check that the answer contains expected concepts. In a full pipeline you would use a test set and metrics (e.g. faithfulness, relevance).

In [7]:
# Example: one test question and expected keywords (Day 4 spirit)
test_question = "What is cocoa's role in Côte d'Ivoire and what are the main growing regions?"
answer = answer_question(test_question)
expected_terms = ["cocoa", "San-Pédro", "Soubré", "production"]  # at least one should appear
found = [t for t in expected_terms if t.lower() in answer.lower()]
print(f"Q: {test_question}\nA: {answer[:400]}...\nExpected one of {expected_terms} -> found: {found}")

Q: What is cocoa's role in Côte d'Ivoire and what are the main growing regions?
A: Cocoa is central to the economy of Côte d'Ivoire, as it is the world's largest producer, supplying about 40% of global production. It is vital for millions of smallholder families in the south and west of the country. The main growing regions for cocoa in Côte d'Ivoire are San-Pédro, Soubré, Abengourou, Divo, Agboville, and the forest belt in the west....
Expected one of ['cocoa', 'San-Pédro', 'Soubré', 'production'] -> found: ['cocoa', 'San-Pédro', 'Soubré', 'production']


## Gradio chat UI

Ask questions in **English** or **French** about agriculture, health, or finance in Côte d'Ivoire and West Africa. The assistant uses the RAG pipeline and answers in your language.

In [9]:
def chat_fn(message, history):
    return answer_question(message)

gr.ChatInterface(
    fn=chat_fn,
    title="Week 5 RAG — West Africa Expert (Côte d'Ivoire)",
    description="Ask about agriculture, health, or finance in Côte d'Ivoire / Afrique de l'Ouest. Posez vos questions en français ou en anglais.",
).launch(theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
